# 02 — Traitement distribué avec PySpark et Spark UI
**TP2 Big Data · BigData Commerce Analytics · B2 Sup de Vinci Rennes**

Chaîne : `MongoDB (RAW) → PySpark → indicateurs préparatoires`

| Partie | Contenu |
|---|---|
| 1 | Vérifier l'environnement et démarrer Spark (`local[*]`) |
| 2 | Lire `raw_commandes` et `raw_payments` dans des DataFrames |
| 3 | Transformations et actions |
| 4 | Lazy evaluation et Spark UI |
| 5 | Partitions, Jobs, Stages, Tasks |
| 6 | Atelier 3 : les 4 indicateurs |
| 7 | Questions de synthèse |

**Prérequis** : notebook 01 exécuté, MongoDB démarré, Java 17+, `pip install pyspark==4.0.1`.

## 1. Préparer PySpark

In [ ]:
import subprocess, sys
print("Python :", sys.version.split()[0])
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)   # Java 17 attendu

import pyspark
print("PySpark :", pyspark.__version__)                                               # 4.0.1 attendu

### Session Spark en mode local

* `master("local[*]")` : Spark tourne sur ta machine avec **tous les cœurs** disponibles.
* `spark.jars.packages` : télécharge le connecteur MongoDB (Maven). Pour Spark 4.0.1 (Scala 2.13) il faut un artefact `_2.13`.
  **Vérifie la version du connecteur** sur Maven Central (`org.mongodb.spark:mongo-spark-connector_2.13`) ou dans la doc MongoDB
  et adapte `MONGO_SPARK_PACKAGE` si elle ne charge pas.
* Spark UI sera disponible sur **http://localhost:4040** tant que la session est ouverte.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pymongo import MongoClient
import pandas as pd

MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "commerce"
MONGO_SPARK_PACKAGE = "org.mongodb.spark:mongo-spark-connector_2.13:10.5.0"   # à vérifier / adapter

spark = (SparkSession.builder
         .appName("BigDataCommerceAnalytics")
         .master("local[*]")
         .config("spark.jars.packages", MONGO_SPARK_PACKAGE)
         .config("spark.mongodb.read.connection.uri", MONGO_URI)
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark UI      :", spark.sparkContext.uiWebUrl)     # normalement http://localhost:4040

## 2. Lire les données depuis MongoDB

La fonction tente d'abord le **connecteur MongoDB** (méthode du TP). Si le connecteur ne se charge pas
(version incompatible, pas d'Internet pour Maven), elle se replie sur `pymongo` + `createDataFrame` afin de ne pas être bloqué.

In [ ]:
def lire_collection(nom):
    try:
        return (spark.read.format("mongodb")
                .option("connection.uri", MONGO_URI)
                .option("database", DB_NAME)
                .option("collection", nom)
                .load())
    except Exception as e:
        print(f"⚠️ Connecteur indisponible ({type(e).__name__}) -> repli pymongo pour {nom}")
        docs = list(MongoClient(MONGO_URI)[DB_NAME][nom].find({}, {"_id": 0}))
        return spark.createDataFrame(pd.DataFrame(docs))

cmd = lire_collection("raw_commandes")
pay = lire_collection("raw_payments")

print("=== raw_commandes ===")
cmd.printSchema()
cmd.show(5, truncate=False)

print("=== raw_payments ===")
pay.printSchema()
pay.show(5, truncate=False)

print("Documents lus  : commandes =", cmd.count(), "| paiements =", pay.count())   # 800 | 208

> **Pourquoi un champ `_id` ?** C'est l'identifiant technique (type `ObjectId`) que MongoDB ajoute automatiquement à chaque document
> qui n'en possède pas. Il est unique, mais n'a pas de sens métier : on l'ignore dans les analyses (ou on le retire avec `.drop("_id")`).
>
> Le schéma est **déduit** par le connecteur en échantillonnant des documents : les types viennent de MongoDB (ex. `order_date` est une chaîne).

## 3. Transformations et actions

| Type | Opérations utilisées ici |
|---|---|
| **Transformations** (lazy, renvoient un DataFrame) | `select`, `filter`, `withColumn`, `groupBy().agg()`, `orderBy`, `join` |
| **Actions** (déclenchent le calcul) | `show`, `count`, `collect`, `take`, `first`, `toPandas` |

In [ ]:
# --- Transformations : sélection, filtre, nouvelle colonne
lignes = (cmd
          .select("cart_id", "user_id", "category", "product_title", "quantity", "unit_price", "discounted_line_total")
          .filter(F.col("quantity") >= 1)
          .withColumn("remise_ligne", F.round(F.col("quantity") * F.col("unit_price") - F.col("discounted_line_total"), 2)))

# --- Transformations : regroupement par catégorie, agrégats, tri
par_categorie = (lignes
                 .groupBy("category")
                 .agg(F.count("*").alias("nb_lignes"),
                      F.sum("quantity").alias("quantite_totale"),
                      F.round(F.sum("discounted_line_total"), 2).alias("ca_categorie"),
                      F.round(F.avg("discounted_line_total"), 2).alias("panier_moyen_ligne"))
                 .orderBy(F.desc("ca_categorie")))

# --- Actions : plusieurs actions déclenchent plusieurs calculs
par_categorie.show(10, truncate=False)          # action 1
print("Nombre de catégories :", par_categorie.count())   # action 2
print("Catégorie n°1 :", par_categorie.first())          # action 3

**À retenir** : tant qu'aucune action n'est appelée, Spark ne fait que *décrire* ce qu'il faudra calculer
(plan logique). Chaque action (`show`, `count`, `first`) lance un **nouveau Job**.

## 4. Lazy evaluation et Spark UI

**Manip à faire dans l'ordre :**
1. Ouvre **http://localhost:4040** → onglet *Jobs*. Note le nombre de jobs déjà présents.
2. Exécute la cellule « transformations seulement » → **rafraîchis Spark UI : aucun nouveau job**.
3. Exécute la cellule « action » → rafraîchis : **un nouveau job apparaît**.

In [ ]:
# Plusieurs transformations successives, SANS action
t = (cmd
     .filter(F.col("discounted_line_total") > 100)
     .withColumn("remise_pct", F.round(100 * (F.col("line_total") - F.col("discounted_line_total")) / F.col("line_total"), 2))
     .filter(F.col("remise_pct") > 5)
     .select("cart_id", "category", "discounted_line_total", "remise_pct")
     .groupBy("category").agg(F.round(F.avg("remise_pct"), 2).alias("remise_moy_pct"))
     .orderBy(F.desc("remise_moy_pct")))

print("✅ Transformations définies. Regarde Spark UI > Jobs : aucun job de plus.")

In [ ]:
# Le plan est construit mais pas exécuté : on peut l'afficher sans lancer de job
t.explain()

In [ ]:
# ACTION : c'est ici que Spark exécute réellement toutes les transformations ci-dessus
t.show()
print("➡️ Rafraîchis Spark UI : un nouveau Job est apparu (onglet Jobs / SQL).")

**Explication (lazy evaluation)** : Spark n'exécute pas les transformations au moment où on les écrit ; il construit un graphe
(DAG). Le calcul démarre seulement quand une **action** est appelée. Avantage : Spark voit toute la chaîne, l'optimise
(Catalyst : filtres poussés au plus tôt, colonnes inutiles supprimées) et évite des calculs intermédiaires inutiles.

## 5. Partitions, Jobs, Stages, Tasks

In [ ]:
print("Partitions de cmd :", cmd.rdd.getNumPartitions())
print("Partitions de pay :", pay.rdd.getNumPartitions())

print("\nRépartition des lignes de cmd par partition :")
(cmd.withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id").count().orderBy("partition_id").show())

In [ ]:
# Observer l'effet d'un changement du nombre de partitions
cmd4 = cmd.repartition(4)
print("Après repartition(4) :", cmd4.rdd.getNumPartitions())
cmd4.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

# Un groupBy provoque un shuffle -> nouveau Stage. Nombre de partitions après shuffle :
print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))
cmd4.groupBy("category").count().count()    # action -> à repérer dans Spark UI

### Observations Spark UI (http://localhost:4040) — à compléter avec tes captures

| Onglet | Ce qu'on y voit | Ton observation |
|---|---|---|
| **Jobs** | 1 job par action (`show`, `count`, …) | nombre de jobs : … |
| **Stages** | un job est découpé en stages aux frontières de *shuffle* (`groupBy`, `orderBy`, `repartition`) | nombre de stages du job `show` : … |
| **Tasks** (dans le détail d'un stage) | 1 task par partition | nombre de tasks : … |
| **SQL / DataFrame** | plan d'exécution physique de chaque requête | opérateurs vus : … |
| **Executors** | en local : un seul executor « driver » | cœurs : … |
| **Environment** | configuration (master, `spark.jars.packages`, versions) | … |

**Lien partitions → Tasks → Stages → Jobs**
* Une **partition** est un morceau de données traité par un cœur.
* Une **Task** = le travail d'un stage sur **une** partition (N partitions ⇒ N tasks dans le stage).
* Un **Stage** = suite d'opérations réalisables sans échange de données entre partitions ; un *shuffle* (`groupBy`, `orderBy`, `repartition`) coupe un stage en deux.
* Un **Job** = tout ce qui est déclenché par **une action** ; il contient un ou plusieurs stages.

> Les temps d'exécution sur ce petit jeu de données en local ne sont pas représentatifs d'un cluster de production.

## 6. Atelier 3 — Indicateurs BigData Commerce Analytics

⚠️ **Piège** : le total du panier est répété sur chaque ligne de produit de `raw_commandes`. On ne l'additionne **jamais** depuis les commandes.
* Le **CA par catégorie** se calcule sur les *lignes* (`discounted_line_total`) : chaque ligne appartient à une seule catégorie.
* Tout ce qui concerne l'argent **encaissé** (top clients, évolution mensuelle) vient de `raw_payments`, filtré sur `status = "PAID"` (1 ligne par panier).

### Indicateur 1 — Chiffre d'affaires par catégorie

In [ ]:
ca_categorie = (cmd.groupBy("category")
                   .agg(F.round(F.sum("discounted_line_total"), 2).alias("ca"))
                   .orderBy(F.desc("ca")))
ca_cat_pd = ca_categorie.toPandas()
ca_cat_pd["part_%"] = (100 * ca_cat_pd["ca"] / ca_cat_pd["ca"].sum()).round(1)
ca_cat_pd.head(10)

### Indicateur 2 — Top 10 clients (paiements PAID par `user_id`)

In [ ]:
# username : une seule valeur par user_id dans les commandes
clients = cmd.select("user_id", "username").dropDuplicates(["user_id"])

top_clients = (pay.filter(F.col("status") == "PAID")
                  .groupBy("user_id")
                  .agg(F.round(F.sum("amount"), 2).alias("total_paye"),
                       F.count("*").alias("nb_paiements_PAID"))
                  .join(clients, "user_id", "left")
                  .orderBy(F.desc("total_paye"))
                  .limit(10))
top_pd = top_clients.toPandas()
top_pd

### Indicateur 3 — Taux de refus

In [ ]:
nb_total   = pay.count()
nb_refuses = pay.filter(F.col("status") == "REFUSED").count()
taux_refus = nb_refuses / nb_total
print(f"Refusés : {nb_refuses} / {nb_total} tentatives  ->  taux de refus = {taux_refus:.1%}")

repartition = (pay.groupBy("status")
                  .agg(F.count("*").alias("nb"), F.round(F.sum("amount"), 2).alias("montant"))
                  .orderBy("status"))
repartition.show()

### Indicateur 4 — Évolution mensuelle des paiements PAID

In [ ]:
mensuel = (pay.filter(F.col("status") == "PAID")
              .withColumn("mois", F.date_format(F.to_date("order_date"), "yyyy-MM"))
              .groupBy("mois")
              .agg(F.round(F.sum("amount"), 2).alias("montant_paid"),
                   F.count("*").alias("nb_paiements"))
              .orderBy("mois"))
mensuel_pd = mensuel.toPandas()
mensuel_pd

### Chiffre d'affaires encaissé et sauvegarde pour le jour 3

In [ ]:
ca_encaisse = pay.filter(F.col("status") == "PAID").agg(F.round(F.sum("amount"), 2)).first()[0]
ca_total_paniers = pay.agg(F.round(F.sum("amount"), 2)).first()[0]
print(f"CA encaissé (PAID)        : {ca_encaisse:,.2f}")
print(f"Montant total des paniers : {ca_total_paniers:,.2f}")

# Contrôle : le CA par catégorie (lignes) doit égaler le total des paniers (comptés une fois)
print(f"CA par catégorie cumulé   : {ca_cat_pd['ca'].sum():,.2f}   (doit être égal au total des paniers)")

from pathlib import Path
Path("data/indicateurs").mkdir(parents=True, exist_ok=True)
ca_cat_pd.to_csv("data/indicateurs/ca_par_categorie.csv", index=False)
top_pd.to_csv("data/indicateurs/top10_clients.csv", index=False)
mensuel_pd.to_csv("data/indicateurs/evolution_mensuelle.csv", index=False)
repartition.toPandas().to_csv("data/indicateurs/repartition_paiements.csv", index=False)
print("Indicateurs exportés dans data/indicateurs/ (réutilisables au jour 3)")

### Commentaire automatique des résultats
La cellule suivante rédige un commentaire à partir de **tes** résultats. Recopie-le dans la cellule Markdown d'après
et reformule-le avec tes mots : le livrable demande un commentaire écrit, pas seulement du code.

In [ ]:
top_cat   = ca_cat_pd.iloc[0]
top3_part = ca_cat_pd["part_%"].head(3).sum()
top_cli   = top_pd.iloc[0]
mois_max  = mensuel_pd.loc[mensuel_pd["montant_paid"].idxmax()]
mois_min  = mensuel_pd.loc[mensuel_pd["montant_paid"].idxmin()]
paid = repartition.toPandas().set_index("status")

print(f"• CA par catégorie : « {top_cat['category']} » domine avec {top_cat['ca']:,.0f} ({top_cat['part_%']} % du total) ; "
      f"les 3 premières catégories pèsent {top3_part:.1f} %.")
print(f"• Top clients : le client {int(top_cli['user_id'])} ({top_cli['username']}) a payé {top_cli['total_paye']:,.0f} ; "
      f"le top 10 représente {100*top_pd['total_paye'].sum()/ca_encaisse:.1f} % du CA encaissé.")
print(f"• Taux de refus : {taux_refus:.1%} (théorique 20 %) ; {int(paid.loc['PENDING','nb'])} paiements restent en attente "
      f"({paid.loc['PENDING','montant']:,.0f} non encaissés).")
print(f"• Évolution mensuelle : pic en {mois_max['mois']} ({mois_max['montant_paid']:,.0f}), creux en {mois_min['mois']} ({mois_min['montant_paid']:,.0f}). "
      f"Attention : order_date est une date synthétique, la saisonnalité n'a donc pas de valeur métier réelle.")
print(f"• CA encaissé : {ca_encaisse:,.0f} soit {100*ca_encaisse/ca_total_paniers:.1f} % de la valeur des paniers.")

### Mon commentaire (à rédiger)

*Écris ici 4 à 6 phrases : quelle catégorie rapporte le plus, concentration du CA sur quelques clients, niveau du taux de refus
par rapport aux 20 % attendus, lecture de la courbe mensuelle (avec la réserve sur les dates synthétiques).*

## 7. Questions de synthèse

**Collection MongoDB vs DataFrame Spark ?**
Une collection MongoDB est un espace de **stockage persistant** de documents JSON/BSON, au schéma flexible, sur le serveur de base de données.
Un DataFrame Spark est une table **distribuée en mémoire**, avec un schéma, découpée en partitions, destinée au **calcul** ; il n'est pas stocké (il est recréé à chaque lecture).

**Transformation vs action ?**
Une transformation (`select`, `filter`, `groupBy`…) décrit un calcul, est paresseuse et renvoie un nouveau DataFrame.
Une action (`show`, `count`, `collect`, `write`…) déclenche l'exécution et renvoie un résultat au driver ou écrit des données.

**Pourquoi « lazy evaluation » ?**
Spark retarde l'exécution jusqu'à une action : il construit d'abord le plan complet, l'optimise (Catalyst), puis l'exécute une seule fois, ce qui évite des calculs et des lectures inutiles.

**Rôle des partitions, Jobs, Stages, Tasks ?**
Les données sont découpées en **partitions** traitées en parallèle. Une **action** crée un **Job**, découpé en **Stages** aux frontières de shuffle ; chaque Stage est exécuté par des **Tasks**, une par partition.

**Pourquoi le CA encaissé = uniquement les paiements PAID ?**
Seul un paiement accepté représente de l'argent réellement reçu. `REFUSED` n'a rien encaissé et `PENDING` n'est pas confirmé (il peut échouer) :
les compter gonflerait artificiellement le chiffre d'affaires. Et il faut partir de `raw_payments` (1 ligne par panier), jamais additionner le total du panier répété sur chaque ligne de produit, sinon on compte plusieurs fois le même panier.

### Pour le jour 3
Conserve la base MongoDB et le dossier `data/indicateurs/`. Réfléchis aux **données personnelles** de `raw_users` (nom, e-mail, adresse, téléphone, carte bancaire de démonstration…) : minimisation, anonymisation / pseudonymisation, RGPD.

> ⚠️ Ne lance la cellule suivante qu'**après** tes captures Spark UI : `spark.stop()` ferme aussi l'interface sur le port 4040.

In [ ]:
# spark.stop()